# Market Basket Analysis
**Internship Project | Data Science | Association Rule Mining**

## Objective
Discover products that are frequently purchased together and translate the findings into practical cross-selling ideas.

## Dataset
UCI Machine Learning Repository — Online Retail, a transaction-level dataset from a UK-based online retailer (2010–2011). The notebook downloads the official ZIP file when run in an internet-enabled environment.

**Methods:** data cleaning, invoice baskets, Apriori frequent itemsets, and association rules (support, confidence, lift).


In [ ]:
# Install dependencies if needed (uncomment in a fresh environment)
# %pip install pandas numpy matplotlib seaborn mlxtend openpyxl

import os, urllib.request, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.frequent_patterns import apriori, association_rules

os.makedirs("outputs", exist_ok=True)
sns.set_theme(style="whitegrid")


## 1. Download and load the official dataset

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
ZIP_PATH = "online_retail.zip"
XLSX_PATH = "Online Retail.xlsx"

if not os.path.exists(XLSX_PATH):
    if not os.path.exists(ZIP_PATH):
        try:
            urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        except Exception as exc:
            raise RuntimeError(
                "Could not download the UCI dataset. Download Online Retail.xlsx from "
                "https://archive.ics.uci.edu/dataset/352/online+retail and upload it beside this notebook."
            ) from exc
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        candidates = [n for n in zf.namelist() if n.lower().endswith(".xlsx")]
        if not candidates:
            raise FileNotFoundError("No .xlsx file found inside the downloaded UCI ZIP.")
        with zf.open(candidates[0]) as src, open(XLSX_PATH, "wb") as dst:
            dst.write(src.read())

df = pd.read_excel(XLSX_PATH)
print("Raw shape:", df.shape)
display(df.head())


## 2. Clean transactions

In [ ]:
df.columns = [c.strip() for c in df.columns]
df["Description"] = df["Description"].astype("string").str.strip()
df["InvoiceNo"] = df["InvoiceNo"].astype("string").str.strip()

# Remove missing product/invoice records and cancellation invoices (InvoiceNo starts with C).
df = df.dropna(subset=["InvoiceNo", "Description", "Quantity", "UnitPrice"])
df = df[~df["InvoiceNo"].str.upper().str.startswith("C", na=False)]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df[~df["Description"].str.lower().str.contains("postage|carriage|discount", na=False)]
df["Revenue"] = df["Quantity"] * df["UnitPrice"]
print("Cleaned rows:", len(df))
print("Unique invoices:", df["InvoiceNo"].nunique())
print("Unique products:", df["Description"].nunique())
display(df[["InvoiceNo", "Description", "Quantity", "UnitPrice", "Revenue"]].head())


## 3. Exploratory analysis

In [ ]:
top_products = df.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(15)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_products.values, y=top_products.index)
plt.title("Top 15 Products by Quantity Sold")
plt.xlabel("Quantity sold")
plt.ylabel("Product")
plt.tight_layout()
plt.savefig("outputs/top_products.png", dpi=160, bbox_inches="tight")
plt.show()

invoice_revenue = df.groupby("InvoiceNo")["Revenue"].sum()
print("Median invoice revenue: £{:.2f}".format(invoice_revenue.median()))
print("Mean invoice revenue: £{:.2f}".format(invoice_revenue.mean()))
print("Unique invoices:", invoice_revenue.size)


## 4. Create one-hot encoded baskets

In [ ]:
# Limit to the United Kingdom to focus the initial analysis on one market.
uk = df[df["Country"].eq("United Kingdom")].copy()

# To keep the Apriori computation efficient in a notebook, select the 80 most frequently
# purchased product descriptions, then retain invoices with at least 2 selected products.
top_n_products = 80
popular_products = uk["Description"].value_counts().head(top_n_products).index
basket_rows = uk[uk["Description"].isin(popular_products)]

basket = (basket_rows.groupby(["InvoiceNo", "Description"])["Quantity"]
          .sum().unstack(fill_value=0))
basket = (basket > 0).astype(bool)
basket = basket.loc[basket.sum(axis=1) >= 2]

print("Basket matrix shape:", basket.shape)
print("Products in matrix:", basket.shape[1])
print("Invoices with at least two selected products:", basket.shape[0])
display(basket.iloc[:5, :10])


## 5. Find frequent itemsets with Apriori

In [ ]:
# Support 1% is a reasonable starting point for this limited, popular-product basket.
# If your environment has limited memory, reduce top_n_products to 40 or increase min_support.
frequent_itemsets = apriori(
    basket, min_support=0.01, use_colnames=True, max_len=3, low_memory=True
)
frequent_itemsets["item_count"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets = frequent_itemsets.sort_values(["support", "item_count"], ascending=[False, False])

print("Frequent itemsets found:", len(frequent_itemsets))
display(frequent_itemsets.head(20))
frequent_itemsets.assign(itemsets=frequent_itemsets["itemsets"].apply(lambda s: ", ".join(sorted(s)))).to_csv(
    "outputs/frequent_itemsets.csv", index=False
)


## 6. Generate and filter association rules

In [ ]:
if frequent_itemsets.empty or frequent_itemsets["item_count"].max() < 2:
    rules = pd.DataFrame()
    print("No multi-item frequent itemsets found. Try lowering min_support to 0.005.")
else:
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.20)
    rules = rules[(rules["lift"] > 1.0) & (rules["antecedents"].apply(len) > 0) & (rules["consequents"].apply(len) > 0)]
    rules = rules.sort_values(["lift", "confidence", "support"], ascending=False)

    def names(itemset):
        return ", ".join(sorted(map(str, itemset)))

    rules["antecedents_text"] = rules["antecedents"].apply(names)
    rules["consequents_text"] = rules["consequents"].apply(names)
    print("Rules after filtering:", len(rules))
    display(rules[["antecedents_text", "consequents_text", "support", "confidence", "lift"]].head(20).round(3))
    rules[["antecedents_text", "consequents_text", "support", "confidence", "lift", "leverage", "conviction"]].to_csv(
        "outputs/association_rules.csv", index=False
    )


## 7. Visualize top association rules

In [ ]:
if not rules.empty:
    plot_rules = rules.head(15).copy()
    plot_rules["Rule"] = plot_rules["antecedents_text"] + " → " + plot_rules["consequents_text"]
    plt.figure(figsize=(11, 7))
    sns.barplot(data=plot_rules, x="lift", y="Rule")
    plt.title("Top Association Rules by Lift")
    plt.xlabel("Lift (values above 1 indicate a positive association)")
    plt.ylabel("Association rule")
    plt.tight_layout()
    plt.savefig("outputs/top_association_rules.png", dpi=160, bbox_inches="tight")
    plt.show()

    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=rules, x="support", y="confidence", size="lift", hue="lift", sizes=(30, 250))
    plt.title("Association Rules: Support vs Confidence")
    plt.tight_layout()
    plt.savefig("outputs/rule_scatter.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("No rules to visualize yet. Lower the minimum support or confidence and rerun the Apriori cells.")


## 8. Business recommendations

Use the rules produced in your run to select a few examples. Interpret the measures carefully:

- **Support:** fraction of analyzed invoices containing all items in a rule.
- **Confidence:** fraction of invoices containing the antecedent that also contain the consequent.
- **Lift:** how much more often the items occur together than expected if independent. Lift above 1 indicates a positive association.

Possible actions to evaluate:
1. Place frequently associated products near each other in online recommendations.
2. Test “frequently bought together” bundles, with margins and inventory constraints checked first.
3. Use high-confidence rules for complementary product suggestions.
4. Run an A/B test before claiming that a recommendation increases sales.

## 9. Limitations
This dataset is historical and comes from one retailer. The notebook focuses on UK invoices and the 80 most frequent products to keep computation manageable. Association does not prove causation, and cancellations/returns are excluded from this analysis.

### Output files
The notebook writes cleaned-analysis summaries, frequent itemsets, association rules, and charts to `outputs/`.
